# Nifty 50 10-Year 1-Minute Export

This notebook drives the Experiment 8 Flattrade client without importing config.py.

- Replace the placeholder credentials below before running any code cell.
- The 1-minute archive writes to this notebook folder as `nifty50_1min_last10y.csv`.
- Adjust `YEARS_OF_HISTORY` if you need a shorter lookback or faster iterations.

In [29]:
import sys
import time
from datetime import datetime, timedelta, time as dtime
from pathlib import Path
import pandas as pd

workspace_root = Path.cwd().resolve()
experiment_root = workspace_root / "scripts" / "claude" / "experiment8"
api_root = experiment_root / "pythonAPI-main" / "pythonAPI-main"
sys.path.insert(0, str(experiment_root))
sys.path.insert(0, str(api_root))
sys.path.insert(0, str(api_root / "dist"))

# === Flattrade credentials (fill in the blanks) ===
USER_ID = "FZ31397"  # e.g., FZ12345
USER_TOKEN = "890365fb384375e17383609e7d0b033e3591add7409dbe3e281dd2f992e8fc26"  # session token obtained via Flattrade login
SPOT_TOKEN = "26000"  # Nifty 50 index token

NOTEBOOK_DIR = (workspace_root / "scripts" / "github_copilot").resolve()
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = NOTEBOOK_DIR / "nifty50_1min_last10y.csv"

YEARS_OF_HISTORY = .25
RATE_LIMIT_SECONDS = 0.55
MARKET_OPEN = dtime(9, 15)
MARKET_CLOSE = dtime(15, 30)

In [30]:
from api_helper import NorenApiPy

def initialize_api():
    if not USER_ID or not USER_TOKEN:
        raise RuntimeError("Set USER_ID and USER_TOKEN in this notebook before running the pipeline.")
    client = NorenApiPy()
    success = client.set_session(userid=USER_ID, password="", usertoken=USER_TOKEN)
    if not success:
        raise RuntimeError("Flattrade session setup failed – verify your token.")
    print(f"Logged in as {USER_ID}")
    return client

api = initialize_api()

Logged in as FZ31397


In [23]:
def trading_calendar(years: int):
    end_date = datetime.now().date()
    start_date = end_date - timedelta(days=365 * years + 30)
    return pd.date_range(start=start_date, end=end_date, freq="B")

def trading_window(day: pd.Timestamp):
    start_dt = datetime.combine(day.date(), MARKET_OPEN)
    end_dt = datetime.combine(day.date(), MARKET_CLOSE)
    return int(start_dt.timestamp()), int(end_dt.timestamp())

def fetch_spot_day(client, day: pd.Timestamp):
    start_ts, end_ts = trading_window(day)
    try:
        response = client.get_time_price_series(exchange="NSE", token=SPOT_TOKEN, starttime=start_ts, endtime=end_ts, interval=1)
    except Exception as exc:
        print(f"   ⚠️ Error fetching {day.date()}: {exc}")
        return None
    if not isinstance(response, list) or not response:
        return None
    df = pd.DataFrame(response)
    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], dayfirst=True)
    numeric_cols = [col for col in ("into", "inth", "intl", "intc", "intv", "intvwap", "v") if col in df.columns]
    if numeric_cols:
        df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    return df.sort_values("time") if "time" in df.columns else df

In [31]:
calendar = trading_calendar(YEARS_OF_HISTORY)
existing_df = None
if OUTPUT_FILE.exists():
    existing_df = pd.read_csv(OUTPUT_FILE)
    if "time" in existing_df.columns:
        existing_df["time"] = pd.to_datetime(existing_df["time"], errors="coerce")
        existing_df.sort_values("time", inplace=True)
        existing_df.drop_duplicates(subset="time", inplace=True)
        print(f"Loaded {len(existing_df)} candles from {OUTPUT_FILE}")
    else:
        print(f"Loaded {len(existing_df)} rows from {OUTPUT_FILE} (no 'time' column)")
processed_dates = set(existing_df["time"].dt.date) if existing_df is not None and "time" in existing_df.columns else set()

new_frames = []
total_days = len(calendar)
for position, day_ts in enumerate(calendar, start=1):
    day = day_ts.date()
    if day in processed_dates:
        continue
    print(f"[{position}/{total_days}] Fetching {day.isoformat()} ...", end="")
    day_df = fetch_spot_day(api, day_ts)
    if day_df is not None and not day_df.empty:
        new_frames.append(day_df)
        print(f" {len(day_df)} rows")
    else:
        print(" no data")
    time.sleep(RATE_LIMIT_SECONDS)

frames_to_concatenate = []
if existing_df is not None and not existing_df.empty:
    frames_to_concatenate.append(existing_df)
frames_to_concatenate.extend(new_frames)

if not frames_to_concatenate:
    print("No candles collected yet.")
else:
    master_df = pd.concat(frames_to_concatenate, ignore_index=True)
    if "time" in master_df.columns:
        master_df.sort_values("time", inplace=True)
        master_df.drop_duplicates(subset="time", inplace=True)
    master_df.to_csv(OUTPUT_FILE, index=False)
    print(f"Saved {len(master_df)} candles to {OUTPUT_FILE}")

[1/86] Fetching 2025-10-06 ... no data
[2/86] Fetching 2025-10-07 ... no data
[3/86] Fetching 2025-10-08 ... no data
[4/86] Fetching 2025-10-09 ... no data
[5/86] Fetching 2025-10-10 ... no data
[6/86] Fetching 2025-10-13 ... no data
[7/86] Fetching 2025-10-14 ... no data
[8/86] Fetching 2025-10-15 ... no data
[9/86] Fetching 2025-10-16 ... no data
[10/86] Fetching 2025-10-17 ... no data
[11/86] Fetching 2025-10-20 ... no data
[12/86] Fetching 2025-10-21 ... no data
[13/86] Fetching 2025-10-22 ... no data
[14/86] Fetching 2025-10-23 ... no data
[15/86] Fetching 2025-10-24 ... no data
[16/86] Fetching 2025-10-27 ... no data
[17/86] Fetching 2025-10-28 ... no data
[18/86] Fetching 2025-10-29 ... no data
[19/86] Fetching 2025-10-30 ... no data
[20/86] Fetching 2025-10-31 ... no data
[21/86] Fetching 2025-11-03 ... no data
[22/86] Fetching 2025-11-04 ... no data
[23/86] Fetching 2025-11-05 ... no data
[24/86] Fetching 2025-11-06 ... no data
[25/86] Fetching 2025-11-07 ... no data
[26/86] F

In [25]:
# Test: Try fetching today's data to verify API is working
from datetime import datetime
today = datetime.now()
test_start = int(datetime.combine(today.date(), MARKET_OPEN).timestamp())
test_end = int(datetime.combine(today.date(), MARKET_CLOSE).timestamp())

print(f"Testing API with today's date: {today.date()}")
print(f"Start timestamp: {test_start}, End timestamp: {test_end}")

try:
    test_response = api.get_time_price_series(
        exchange="NSE", 
        token=SPOT_TOKEN, 
        starttime=test_start, 
        endtime=test_end, 
        interval=1
    )
    print(f"Response type: {type(test_response)}")
    print(f"Response: {test_response[:3] if isinstance(test_response, list) and len(test_response) > 0 else test_response}")
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

Testing API with today's date: 2026-02-02
Start timestamp: 1770003900, End timestamp: 1770026400
Response type: <class 'NoneType'>
Response: None


In [26]:
# Test: Try fetching without time constraints (should return latest available data)
print("Testing API without starttime/endtime parameters...")
try:
    test_response = api.get_time_price_series(
        exchange="NSE", 
        token=SPOT_TOKEN,
        starttime=None,
        endtime=None,
        interval=1
    )
    if test_response:
        print(f"✅ Got {len(test_response)} candles")
        print(f"First candle: {test_response[0]}")
        print(f"Last candle: {test_response[-1]}")
        
        # Convert to DataFrame to see structure
        test_df = pd.DataFrame(test_response)
        print(f"\nDataFrame shape: {test_df.shape}")
        print(f"Columns: {test_df.columns.tolist()}")
        print(f"\nFirst few rows:")
        print(test_df.head())
    else:
        print("❌ Response is None")
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

Testing API without starttime/endtime parameters...
❌ Response is None


In [27]:
# Test: Try getting quotes (real-time) instead of time series
print("Testing get_quotes API (should work if session is valid)...")
try:
    quote_response = api.get_quotes(exchange="NSE", token=SPOT_TOKEN)
    if quote_response:
        print(f"✅ Quote API works!")
        print(f"Response keys: {quote_response.keys()}")
        print(f"Last Price (lp): {quote_response.get('lp', 'N/A')}")
        print(f"Full response: {quote_response}")
    else:
        print("❌ Quote response is None")
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)
print("Note: get_time_price_series might not have historical data")
print("Flattrade API may only provide data from current trading day")
print("For 10 years of data, you may need to use get_daily_price_series")
print("or a different data source.")
print("="*60)

Testing get_quotes API (should work if session is valid)...
❌ Quote response is None

Note: get_time_price_series might not have historical data
Flattrade API may only provide data from current trading day
For 10 years of data, you may need to use get_daily_price_series
or a different data source.


In [28]:
# Check current date info and try daily price series
from datetime import datetime, timedelta
import time as time_module

today = datetime.now()
print(f"Today: {today.strftime('%Y-%m-%d %A')}")
print(f"Market might be closed if it's weekend/holiday\n")

# Try get_daily_price_series which should have more historical data
print("Testing get_daily_price_series (EOD data)...")
try:
    # Get last 30 days of daily data
    end_date = datetime.now()
    start_date = end_date - timedelta(days=30)
    
    daily_response = api.get_daily_price_series(
        exchange="NSE",
        tradingsymbol="NIFTY 50",
        startdate=int(start_date.timestamp()),
        enddate=int(end_date.timestamp())
    )
    
    if daily_response:
        print(f"✅ Got {len(daily_response)} daily candles")
        print(f"First candle: {daily_response[0]}")
        print(f"Last candle: {daily_response[-1]}")
        
        # Convert to DataFrame
        daily_df = pd.DataFrame(daily_response)
        print(f"\nDataFrame columns: {daily_df.columns.tolist()}")
        print(f"\nLast 5 days:")
        print(daily_df.tail())
    else:
        print("❌ Daily price series also returned None")
        print("\nPossible issues:")
        print("1. Symbol name might be incorrect (try 'Nifty 50' or 'NIFTY50')")
        print("2. Token might not have historical data access")
        print("3. This endpoint might not support index data")
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

Today: 2026-02-02 Monday
Market might be closed if it's weekend/holiday

Testing get_daily_price_series (EOD data)...
❌ Daily price series also returned None

Possible issues:
1. Symbol name might be incorrect (try 'Nifty 50' or 'NIFTY50')
2. Token might not have historical data access
3. This endpoint might not support index data
